In [ ]:
import os
import getpass
from dotenv import load_dotenv

import json
import copy
import pandas as pd
import numpy as np
import re
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional

In [6]:
import json
import os

CUAD_PATH = "../datasets/CUAD_v1/CUAD_v1.json"
SAMPLE_CONTRACT_TITLE = "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT"
OUTPUT_FILENAME = "../outputs/prototype_ui_data.json"

def extract_prototype_data(cuad_json_path, target_contract_title):
    print(f"Loading dataset from: {cuad_json_path}...")
    
    # 1. Load the SQuAD JSON
    try:
        with open(cuad_json_path, 'r', encoding='utf-8') as f:
            cuad_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find the file at {cuad_json_path}")
        return None

    # 2. Find our specific single contract
    contract = next((c for c in cuad_data['data'] if c['title'] == target_contract_title), None)
    
    if not contract:
        print(f"Error: Contract '{target_contract_title}' not found.")
        return None

    paragraph = contract['paragraphs'][0]
    
    # 3. Initialize our flattened UI JSON structure
    ui_json = {
        "documentId": target_contract_title,
        "rawText": paragraph['context'],
        "nodes": []
    }

    node_counter = 1

    # 4. Loop through the Q&A pairs to build our nodes
    for qa in paragraph['qas']:
        # If is_impossible is True, the clause doesn't exist in this contract. Skip it.
        if not qa['is_impossible']:
            
            # The category is usually appended to the ID (e.g., "...__Parties")
            category = qa['id'].split('__')[-1]

            # A single category might have multiple answers/excerpts
            for answer in qa['answers']:
                text_content = ' '.join(answer['text'].split())  # Normalize whitespace
                # Get original start/end from raw text
                raw_text = answer['text']
                start_index = answer['answer_start']
                end_index = start_index + len(raw_text)
                
                ui_json["nodes"].append({
                    "id": f"node-{node_counter}",
                    "category": category,
                    "extractedText": text_content,  # Cleaned version
                    "textStartIndex": start_index,   # Original indices
                    "textEndIndex": end_index
                })
                node_counter += 1

    return ui_json

# Run the extraction
prototype_data = extract_prototype_data(CUAD_PATH, SAMPLE_CONTRACT_TITLE)

# Save the output if extraction was successful
if prototype_data:
    with open(OUTPUT_FILENAME, "w", encoding="utf-8") as outfile:
        json.dump(prototype_data, outfile, indent=2)
        print(f"✓ Success! Extracted {len(prototype_data['nodes'])} nodes.")
        print(f"✓ Data saved to {OUTPUT_FILENAME}")

Loading dataset from: ../datasets/CUAD_v1/CUAD_v1.json...
✓ Success! Extracted 46 nodes.
✓ Data saved to ../outputs/prototype_ui_data.json
